# Exploração do pipeline RAG — Shopping de Preços

Notebook didático para inspecionar cada etapa do pipeline **sem rede e sem chaves**
(providers `fake` + banco `memory` + fixtures HTML reais dos testes).

> Rode a partir da raiz do projeto: `jupyter lab notebooks/01_exploracao_rag.ipynb`

In [ ]:
# Ambiente de demonstração (antes de importar app.*)
import os, sys
sys.path.insert(0, '..') if 'notebooks' in os.getcwd() else None
os.environ.update(VECTOR_BACKEND='memory', EMBEDDING_PROVIDER='fake',
                  LLM_PROVIDER='fake', RETRIEVAL_MODE='hybrid', DATA_DIR='data-nb')

## 1) Extração: HTML → Markdown
A sidebar cheia de links é podada; código vira bloco *fenced* atômico.

In [ ]:
from tests.conftest import APIDOC_HTML, make_pages
from app.ingestion.extractor import html_to_markdown

doc = html_to_markdown(APIDOC_HTML, 'https://apidoc.shoppingdeprecos.com.br/atualiza-estoque-produto')
print('título:', doc.title, '| updated_at:', doc.updated_at)
print(doc.markdown[:600])

## 2) Metadados + chunking semântico
Note o breadcrumb `[Contexto: ...]` e que o bloco de código não é dividido.

In [ ]:
from app.ingestion.enricher import build_metadata
from app.rag.chunking import SemanticChunker

page = make_pages()[0]
meta = build_metadata(page, doc)
print({k: meta[k] for k in ('endpoint','method','category','section')})

chunks = SemanticChunker(chunk_size=300, chunk_overlap=50).chunk(doc.markdown, meta)
for c in chunks:
    print('---', c.id[:12], '| tokens aprox:', len(c.text)//4)
    print(c.text[:160].replace('\n', ' '), '...')

## 3) Ingestão completa (fixtures) + retrieval híbrido

In [ ]:
from app import container
container.reset_container()
stats = container.get_pipeline().run(full=True, pages=make_pages())
container.refresh_after_ingest()
print(stats.as_dict())

In [ ]:
retriever = container.get_retriever()
for sc in retriever.retrieve('como atualizar estoque de um produto', k=3):
    print(round(sc.score, 4), '|', sc.chunk.metadata['title'], '|', sc.chunk.metadata['url'])

## 4) Pergunta ponta a ponta (cadeia RAG)
Com `LLM_PROVIDER=fake` a resposta é determinística, mas o fluxo (retrieval → contexto → fontes → confiança) é o mesmo da produção.

In [ ]:
chain = container.get_chain()
res = chain.answer('Como autenticar na API do Shopping de Preços?', session_id='nb')
print(res.answer)
print('\nconfiança:', res.confidence, '| latência(ms):', res.latency_ms)

## 5) Fallback anti-alucinação
Pergunta fora da documentação → frase exata, sem fontes.

In [ ]:
res = chain.answer('Qual a previsão do tempo em Marília?', session_id='nb')
print(res.answer)
print('fallback?', res.used_fallback, '| fontes:', res.sources)

## Próximos passos
- Troque os providers no `.env` (OpenAI/Anthropic/E5) e rode `scripts/ingest.py --full` para indexar a documentação real.
- Compare `RETRIEVAL_MODE=vector|bm25|hybrid` e `CHUNK_SIZE=300|500|800|1000` medindo com `scripts/evaluate.py`.